# Customer Churn Prediction V3 - Ultimate Solution
## Playground Series S6E3

---

**Competition Overview:**
| Metric | Value |
|--------|-------|
| Current Score (V1) | 0.91404 |
| Target Score | 0.917+ |
| Expected Score | 0.916-0.918 |

**Strategy:** Conservative (Low Risk + Safe Medium)

**V3 Improvements:**
- ✅ Original dataset integration (Telco Customer Churn)
- ✅ Comprehensive feature engineering (95+ features)
- ✅ Stronger regularization (fix overfitting)
- ✅ Multi-model ensemble (XGBoost + LightGBM + CatBoost + MLP)
- ✅ Multi-seed training (3 seeds)
- ✅ Advanced ensembling (stacking + rank averaging)
- ✅ GPU acceleration for CatBoost/LightGBM (XGBoost uses CPU)

**Expected Runtime:** ~2 hours

---
**Author:** Pratik | **Version:** 3.0 | **Date:** 2026-03-26

## SECTION 1: SETUP & PATH DETECTION

This section automatically detects whether we're running on Kaggle or locally
and sets appropriate paths for data loading.

In [ ]:
# Auto-detect environment and set paths
import os
import gc

# Check if running on Kaggle
if os.path.exists('/kaggle/input'):
    # Running on Kaggle
    DATA_DIR = '/kaggle/input/competitions/playground-series-s6e3'
    OUTPUT_DIR = '/kaggle/working'
    
    # Try to find original data
    possible_origins = [
        '/kaggle/input/telco-customer-churn',
        '/kaggle/input/ibm-telco-customer-churn',
    ]
    ORIGINAL_DIR = None
    for path in possible_origins:
        if os.path.exists(path):
            ORIGINAL_DIR = path
            break
    
    print("Running on Kaggle")
    print(f"   Data directory: {DATA_DIR}")
    print(f"   Original data: {ORIGINAL_DIR if ORIGINAL_DIR else 'Not found'}")
    print(f"   Output directory: {OUTPUT_DIR}")
    
else:
    # Running locally
    DATA_DIR = './data'
    OUTPUT_DIR = './output'
    ORIGINAL_DIR = './original_data'
    
    print("Running locally")
    print(f"   Data directory: {DATA_DIR}")
    print(f"   Original data: {ORIGINAL_DIR if os.path.exists(ORIGINAL_DIR) else 'Not found'}")
    print(f"   Output directory: {OUTPUT_DIR}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

## SECTION 2: IMPORTS

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from scipy.stats import rankdata
from scipy.optimize import minimize

# ML Libraries
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Configuration
SEED = 42
N_FOLDS = 5
SEEDS = [42, 123, 2024]  # Multi-seed ensemble

print(f"XGBoost version: {xgb.__version__}")
print(f"LightGBM version: {lgb.__version__}")
print(f"Optuna version: {optuna.__version__}")
print(f"Seeds: {SEEDS}")
print(f"Folds: {N_FOLDS}")

## SECTION 3: LOAD ALL DATA

In [ ]:
# Load synthetic data (competition data)
print("Loading synthetic data...")
train_syn = pd.read_csv(f'{DATA_DIR}/train.csv')
test_syn = pd.read_csv(f'{DATA_DIR}/test.csv')
sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

# Convert Churn target to binary
train_syn['Churn'] = train_syn['Churn'].map({'Yes': 1, 'No': 0}).fillna(train_syn['Churn']).astype(int)

print(f"   Synthetic train: {train_syn.shape}")
print(f"   Synthetic test: {test_syn.shape}")
print(f"   Target distribution: {train_syn['Churn'].value_counts(normalize=True).to_dict()}")

In [ ]:
# Load original data (Telco Customer Churn)
if ORIGINAL_DIR and os.path.exists(ORIGINAL_DIR):
    print("\nLoading original data...")
    
    # Try different possible filenames
    possible_files = [
        'WA_Fn-UseC_-Telco-Customer-Churn.csv',
        'Telco-Customer-Churn.csv',
        'original.csv',
        'churn.csv'
    ]
    
    original = None
    for filename in possible_files:
        filepath = os.path.join(ORIGINAL_DIR, filename)
        if os.path.exists(filepath):
            original = pd.read_csv(filepath)
            print(f"   Found: {filename}")
            break
    
    if original is not None:
        original['Churn'] = original['Churn'].map({'Yes': 1, 'No': 0}).fillna(original['Churn']).astype(int)
        
        column_mapping = {
            'customerID': 'id',
            'SeniorCitizen': 'SeniorCitizen',
            'Partner': 'Partner',
            'Dependents': 'Dependents',
            'tenure': 'tenure',
            'PhoneService': 'PhoneService',
            'MultipleLines': 'MultipleLines',
            'InternetService': 'InternetService',
            'OnlineSecurity': 'OnlineSecurity',
            'OnlineBackup': 'OnlineBackup',
            'DeviceProtection': 'DeviceProtection',
            'TechSupport': 'TechSupport',
            'StreamingTV': 'StreamingTV',
            'StreamingMovies': 'StreamingMovies',
            'Contract': 'Contract',
            'PaperlessBilling': 'PaperlessBilling',
            'PaymentMethod': 'PaymentMethod',
            'MonthlyCharges': 'MonthlyCharges',
            'TotalCharges': 'TotalCharges'
        }
        
        keep_cols = [c for c in column_mapping.keys() if c in original.columns]
        original = original[keep_cols].rename(columns=column_mapping)
        original['id'] = range(len(original))
        original['source'] = 'original'
        train_syn['source'] = 'synthetic'
        
        print(f"   Original train: {original.shape}")
        USE_ORIGINAL = True
    else:
        print("   Could not find original data file")
        USE_ORIGINAL = False
else:
    print("\nOriginal data directory not found - using synthetic data only")
    USE_ORIGINAL = False

In [ ]:
# Merge synthetic and original data
if USE_ORIGINAL:
    print("\nMerging synthetic + original data...")
    
    common_cols = [c for c in train_syn.columns if c in original.columns]
    train_combined = pd.concat([train_syn[common_cols], original[common_cols]], ignore_index=True)
    
    key_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
                'PhoneService', 'InternetService', 'Contract', 'MonthlyCharges', 'Churn']
    
    before_dedup = len(train_combined)
    train_combined = train_combined.drop_duplicates(subset=key_cols, keep='first')
    after_dedup = len(train_combined)
    
    print(f"   Before deduplication: {before_dedup:,}")
    print(f"   After deduplication: {after_dedup:,}")
    print(f"   Removed: {before_dedup - after_dedup:,} duplicates")
    
    train = train_combined
    test = test_syn
    
    print(f"\nFinal training data: {train.shape}")
    print(f"   Using original data: YES (+{len(original):,} samples)")
else:
    train = train_syn
    test = test_syn
    print(f"\nFinal training data: {train.shape}")
    print(f"   Using original data: NO")

print(f"\nFinal Target Distribution:")
print(train['Churn'].value_counts(normalize=True))

## SECTION 4: COMPREHENSIVE FEATURE ENGINEERING

In [ ]:
def comprehensive_feature_engineering(df, is_train=True, stats_dict=None):
    df = df.copy()
    
    # TENURE FEATURES
    df['tenure_bucket'] = pd.cut(df['tenure'], bins=[-1, 6, 12, 24, 48, 72, 999],
                                labels=[0, 1, 2, 3, 4, 5]).astype(int)
    df['is_new_customer'] = (df['tenure'] <= 6).astype(int)
    df['is_long_term'] = (df['Contract'] != 'Month-to-month').astype(int)
    df['tenure_normalized'] = df['tenure'] / 72.0
    df['tenure_sq'] = df['tenure'] ** 2
    df['tenure_log'] = np.log1p(df['tenure'])
    
    # CHARGE FEATURES
    df['charges_per_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)
    df['total_monthly_ratio'] = df['TotalCharges'] / (df['MonthlyCharges'] + 1)
    df['charges_per_tenure_sq'] = df['charges_per_tenure'] ** 2
    df['expected_total_charges'] = df['MonthlyCharges'] * df['tenure']
    df['charges_deviation'] = df['TotalCharges'] - df['expected_total_charges']
    df['charges_deviation_pct'] = df['charges_deviation'] / (df['expected_total_charges'] + 1)
    
    # Z-score within contract type
    if is_train:
        contract_stats = df.groupby('Contract')['MonthlyCharges'].agg(['mean', 'std', 'median']).reset_index()
        contract_stats.columns = ['Contract', 'contract_mean', 'contract_std', 'contract_median']
        stats_dict = {'contract_stats': contract_stats}
    
    if stats_dict and 'contract_stats' in stats_dict:
        df = df.merge(stats_dict['contract_stats'], on='Contract', how='left')
        df['monthly_charge_z'] = (df['MonthlyCharges'] - df['contract_mean']) / (df['contract_std'] + 1e-6)
        df['monthly_charge_vs_median'] = df['MonthlyCharges'] - df['contract_median']
        df.drop(columns=['contract_mean', 'contract_std', 'contract_median'], inplace=True)
    
    # SERVICE FEATURES
    service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                    'TechSupport', 'StreamingTV', 'StreamingMovies']
    
    for col in service_cols:
        if col in df.columns:
            df[f'{col}_flag'] = (df[col] == 'Yes').astype(int) if df[col].dtype == 'object' else 0
    
    flag_cols = [f'{c}_flag' for c in service_cols if f'{c}_flag' in df.columns]
    df['num_services'] = df[flag_cols].sum(axis=1)
    df['has_all_services'] = (df['num_services'] == len(service_cols)).astype(int)
    df['no_services'] = (df['num_services'] == 0).astype(int)
    df['avg_charge_per_service'] = df['MonthlyCharges'] / (df['num_services'] + 1)
    
    df['security_bundle'] = ((df.get('OnlineSecurity_flag', 0) == 1) & (df.get('TechSupport_flag', 0) == 1)).astype(int)
    df['streaming_bundle'] = ((df.get('StreamingTV_flag', 0) == 1) & (df.get('StreamingMovies_flag', 0) == 1)).astype(int)
    df['protection_bundle'] = ((df.get('OnlineBackup_flag', 0) == 1) & (df.get('DeviceProtection_flag', 0) == 1)).astype(int)
    
    # RISK FLAGS
    charge_median = df['MonthlyCharges'].median() if is_train else 70.0
    
    df['high_risk_flag'] = ((df['Contract'] == 'Month-to-month') & (df['tenure'] <= 12) & (df['MonthlyCharges'] > charge_median)).astype(int)
    df['high_risk_electronic'] = ((df['Contract'] == 'Month-to-month') & (df['PaymentMethod'] == 'Electronic check')).astype(int)
    df['fiber_high_charge'] = ((df['InternetService'] == 'Fiber optic') & (df['MonthlyCharges'] > charge_median)).astype(int)
    df['no_security_high_charge'] = ((df.get('OnlineSecurity_flag', 0) == 0) & (df['MonthlyCharges'] > charge_median)).astype(int)
    df['new_fiber_no_security'] = ((df['tenure'] <= 6) & (df['InternetService'] == 'Fiber optic') & (df.get('OnlineSecurity_flag', 0) == 0)).astype(int)
    
    # FAMILY FEATURES
    if 'Partner' in df.columns:
        df['has_partner'] = (df['Partner'] == 'Yes').astype(int) if df['Partner'].dtype == 'object' else 0
    if 'Dependents' in df.columns:
        df['has_dependents'] = (df['Dependents'] == 'Yes').astype(int) if df['Dependents'].dtype == 'object' else 0
    df['has_family'] = ((df.get('has_partner', 0) == 1) | (df.get('has_dependents', 0) == 1)).astype(int)
    df['single_no_dependents'] = ((df.get('has_partner', 0) == 0) & (df.get('has_dependents', 0) == 0)).astype(int)
    
    # INTERACTION FEATURES
    df['tenure_contract_interaction'] = df['tenure_bucket'] * df['is_long_term']
    df['tenure_services_interaction'] = df['tenure'] * df['num_services']
    df['charges_services_ratio'] = df['MonthlyCharges'] / (df['num_services'] + 1)
    df['tenure_fiber'] = df['tenure'] * (df['InternetService'] == 'Fiber optic').astype(int)
    df['tenure_x_monthly'] = df['tenure'] * df['MonthlyCharges']
    df['charges_x_services'] = df['MonthlyCharges'] * df['num_services']
    
    # CATEGORICAL COMBINATIONS
    df['Contract_Internet'] = df['Contract'].astype(str) + '_' + df['InternetService'].astype(str)
    df['Contract_Payment'] = df['Contract'].astype(str) + '_' + df['PaymentMethod'].astype(str)
    df['Internet_Security'] = df['InternetService'].astype(str) + '_' + df['OnlineSecurity'].astype(str)
    df['Internet_Payment'] = df['InternetService'].astype(str) + '_' + df['PaymentMethod'].astype(str)
    df['Paperless_Payment'] = df['PaperlessBilling'].astype(str) + '_' + df['PaymentMethod'].astype(str)
    
    # PAYMENT BEHAVIOR
    df['is_electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
    df['is_automatic_payment'] = df['PaymentMethod'].str.contains('automatic', case=False, na=False).astype(int)
    df['is_paperless'] = (df['PaperlessBilling'] == 'Yes').astype(int) if df['PaperlessBilling'].dtype == 'object' else 0
    df['paperless_electronic'] = (df['is_paperless'] & df['is_electronic_check']).astype(int)
    
    return df, stats_dict

# Apply feature engineering
print("Applying comprehensive feature engineering...")
train_fe, stats_dict = comprehensive_feature_engineering(train, is_train=True)
test_fe, _ = comprehensive_feature_engineering(test, is_train=False, stats_dict=stats_dict)

original_features = train.columns.tolist()
new_features = [c for c in train_fe.columns if c not in original_features]

print(f"   Original features: {len(original_features)}")
print(f"   New features: {len(new_features)}")
print(f"   Total features: {len(train_fe.columns)}")

## SECTION 5: TARGET & FREQUENCY ENCODING

In [ ]:
def target_encode_kfold(train_df, test_df, cat_cols, target_col, n_splits=5, seed=42):
    train_encoded = train_df.copy()
    test_encoded = test_df.copy()
    
    for col in cat_cols:
        global_mean = train_df[target_col].mean()
        train_encoded[f'{col}_target_enc'] = 0.0
        
        kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        
        for train_idx, val_idx in kf.split(train_df, train_df[target_col]):
            means = train_df.iloc[train_idx].groupby(col)[target_col].mean()
            train_encoded.loc[val_idx, f'{col}_target_enc'] = train_df.iloc[val_idx][col].map(means)
        
        train_encoded[f'{col}_target_enc'] = train_encoded[f'{col}_target_enc'].fillna(global_mean)
        
        test_means = train_df.groupby(col)[target_col].mean()
        test_encoded[f'{col}_target_enc'] = test_df[col].map(test_means).fillna(global_mean)
    
    return train_encoded, test_encoded

def frequency_encode(train_df, test_df, cat_cols):
    train_encoded = train_df.copy()
    test_encoded = test_df.copy()
    
    for col in cat_cols:
        freq_map = train_df[col].value_counts(normalize=True).to_dict()
        train_encoded[f'{col}_freq'] = train_df[col].map(freq_map)
        test_encoded[f'{col}_freq'] = test_df[col].map(freq_map).fillna(0)
    
    return train_encoded, test_encoded

# Identify categorical columns for encoding
cat_cols_for_encoding = train_fe.select_dtypes(include='object').columns.tolist()
cat_cols_for_encoding = [c for c in cat_cols_for_encoding if c not in ['id', 'source']]

print(f"Applying target encoding to {len(cat_cols_for_encoding)} categorical columns...")
train_fe, test_fe = target_encode_kfold(train_fe, test_fe, cat_cols_for_encoding, 'Churn', n_splits=5)

print(f"Applying frequency encoding...")
train_fe, test_fe = frequency_encode(train_fe, test_fe, cat_cols_for_encoding)

print(f"Total features after encoding: {len(train_fe.columns)}")

In [ ]:
# Label encode remaining categoricals
label_encoders = {}
cat_cols_remaining = train_fe.select_dtypes(include='object').columns.tolist()
cat_cols_remaining = [c for c in cat_cols_remaining if c not in ['id', 'source']]

for col in cat_cols_remaining:
    le = LabelEncoder()
    combined = pd.concat([train_fe[col].astype(str), test_fe[col].astype(str)])
    le.fit(combined)
    train_fe[col] = le.transform(train_fe[col].astype(str))
    test_fe[col] = le.transform(test_fe[col].astype(str))
    label_encoders[col] = le

print(f"Label encoded {len(cat_cols_remaining)} categorical columns")

# Prepare final feature matrices
target = 'Churn'
drop_cols = ['id', 'Churn', 'source']
feature_cols = [c for c in train_fe.columns if c not in drop_cols]

X = train_fe[feature_cols].values.astype(np.float32)
y = train_fe[target].values
X_test = test_fe[feature_cols].values.astype(np.float32)

print(f"\nFinal Feature Summary:")
print(f"   Number of features: {len(feature_cols)}")
print(f"   Training samples: {X.shape[0]:,}")
print(f"   Test samples: {X_test.shape[0]:,}")

## SECTION 6: OPTUNA HYPERPARAMETER TUNING

In [ ]:
N_TRIALS = 30

# XGBoost - using CPU (Kaggle XGBoost doesn't support GPU)
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'max_depth': trial.suggest_int('max_depth', 4, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),
        'min_child_weight': trial.suggest_int('min_child_weight', 10, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10, log=True),
        'gamma': trial.suggest_float('gamma', 0.5, 5),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 3, 4),
        'random_state': SEED,
        'eval_metric': 'auc',
        'tree_method': 'hist',  # CPU method
    }
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        model = xgb.XGBClassifier(**params)
        model.fit(X[train_idx], y[train_idx], eval_set=[(X[val_idx], y[val_idx])], verbose=0)
        preds = model.predict_proba(X[val_idx])[:, 1]
        scores.append(roc_auc_score(y[val_idx], preds))
    
    return np.mean(scores)

print("Starting XGBoost Optuna tuning (30 trials)...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest XGBoost AUC: {study_xgb.best_value:.6f}")
print(f"   Best params: {study_xgb.best_params}")

In [ ]:
# LightGBM with GPU
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 30, 100),
        'subsample': trial.suggest_float('subsample', 0.7, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),
        'min_child_samples': trial.suggest_int('min_child_samples', 30, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10, log=True),
        'random_state': SEED,
        'verbose': -1,
        'device': 'gpu',  # GPU
    }
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        model = lgb.LGBMClassifier(**params)
        model.fit(X[train_idx], y[train_idx], eval_set=[(X[val_idx], y[val_idx])], 
                 callbacks=[lgb.log_evaluation(0)])
        preds = model.predict_proba(X[val_idx])[:, 1]
        scores.append(roc_auc_score(y[val_idx], preds))
    
    return np.mean(scores)

print("Starting LightGBM Optuna tuning (30 trials, GPU)...")
study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest LightGBM AUC: {study_lgb.best_value:.6f}")
print(f"   Best params: {study_lgb.best_params}")

In [ ]:
# CatBoost with GPU
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 2000),
        'depth': trial.suggest_int('depth', 4, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 3, 20, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 30, 150),
        'bootstrap_type': 'MVS',  # Required for subsample on GPU
        'bootstrap_type': 'MVS',  # Required for subsample on GPU
        'subsample': trial.suggest_float('subsample', 0.7, 0.9),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 0.8),
        'random_seed': SEED,
        'verbose': 0,
        'eval_metric': 'AUC',
        'task_type': 'GPU',  # GPU
        'devices': '0',
    }
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        model = CatBoostClassifier(**params)
        model.fit(X[train_idx], y[train_idx], eval_set=(X[val_idx], y[val_idx]), verbose=0)
        preds = model.predict_proba(X[val_idx])[:, 1]
        scores.append(roc_auc_score(y[val_idx], preds))
    
    return np.mean(scores)

print("Starting CatBoost Optuna tuning (30 trials, GPU)...")
study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest CatBoost AUC: {study_cat.best_value:.6f}")
print(f"   Best params: {study_cat.best_params}")

In [ ]:
# Scale features for MLP
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# MLP Neural Network
def objective_mlp(trial):
    n_layers = trial.suggest_int('n_layers', 2, 4)
    layers = []
    for i in range(n_layers):
        layers.append(trial.suggest_int(f'n_units_l{i}', 32, 256))
    
    params = {
        'hidden_layer_sizes': tuple(layers),
        'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
        'solver': trial.suggest_categorical('solver', ['adam']),
        'alpha': trial.suggest_float('alpha', 1e-4, 1e-1, log=True),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
        'max_iter': 300,
        'early_stopping': True,
        'validation_fraction': 0.1,
        'random_state': SEED,
    }
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
        model = MLPClassifier(**params)
        model.fit(X_scaled[train_idx], y[train_idx])
        preds = model.predict_proba(X_scaled[val_idx])[:, 1]
        scores.append(roc_auc_score(y[val_idx], preds))
    
    return np.mean(scores)

print("Starting MLP Optuna tuning (20 trials)...")
study_mlp = optuna.create_study(direction='maximize')
study_mlp.optimize(objective_mlp, n_trials=20, show_progress_bar=True)

print(f"\nBest MLP AUC: {study_mlp.best_value:.6f}")
print(f"   Best params: {study_mlp.best_params}")

## SECTION 7: MULTI-SEED TRAINING

In [ ]:
# Base parameters from Optuna
xgb_params_base = {
    **study_xgb.best_params,
    'eval_metric': 'auc',
    'tree_method': 'hist',  # CPU
}

lgb_params_base = {
    **study_lgb.best_params,
    'verbose': -1,
    'device': 'gpu',  # GPU
}

cat_params_base = {
    **study_cat.best_params,
    'verbose': 0,
    'eval_metric': 'AUC',
    'task_type': 'GPU',  # GPU
    'devices': '0',
}

# MLP params
mlp_params = study_mlp.best_params.copy()
mlp_params['hidden_layer_sizes'] = tuple([mlp_params.pop(f'n_units_l{i}') for i in range(mlp_params.pop('n_layers'))])
mlp_params.update({
    'max_iter': 300,
    'early_stopping': True,
    'validation_fraction': 0.1,
})

print(f"Training with {len(SEEDS)} seeds x 4 models = {len(SEEDS)*4} model variants")

# Initialize arrays
n_seeds = len(SEEDS)
oof_xgb = np.zeros((n_seeds, len(X)))
oof_lgb = np.zeros((n_seeds, len(X)))
oof_cat = np.zeros((n_seeds, len(X)))
oof_mlp = np.zeros((n_seeds, len(X)))

test_preds_xgb = np.zeros((n_seeds, len(X_test)))
test_preds_lgb = np.zeros((n_seeds, len(X_test)))
test_preds_cat = np.zeros((n_seeds, len(X_test)))
test_preds_mlp = np.zeros((n_seeds, len(X_test)))

# Training loop
for seed_idx, seed in enumerate(SEEDS):
    print(f"\nSEED {seed} ({seed_idx+1}/{n_seeds})")
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        # XGBoost
        xgb_params = {**xgb_params_base, 'random_state': seed}
        xgb_model = xgb.XGBClassifier(**xgb_params)
        xgb_model.fit(X[train_idx], y[train_idx], eval_set=[(X[val_idx], y[val_idx])], verbose=0)
        oof_xgb[seed_idx, val_idx] = xgb_model.predict_proba(X[val_idx])[:, 1]
        test_preds_xgb[seed_idx] += xgb_model.predict_proba(X_test)[:, 1] / N_FOLDS
        
        # LightGBM
        lgb_params = {**lgb_params_base, 'random_state': seed}
        lgb_model = lgb.LGBMClassifier(**lgb_params)
        lgb_model.fit(X[train_idx], y[train_idx], eval_set=[(X[val_idx], y[val_idx])], 
                     callbacks=[lgb.log_evaluation(0)])
        oof_lgb[seed_idx, val_idx] = lgb_model.predict_proba(X[val_idx])[:, 1]
        test_preds_lgb[seed_idx] += lgb_model.predict_proba(X_test)[:, 1] / N_FOLDS
        
        # CatBoost
        cat_params = {**cat_params_base, 'random_seed': seed}
        cat_model = CatBoostClassifier(**cat_params)
        cat_model.fit(X[train_idx], y[train_idx], eval_set=(X[val_idx], y[val_idx]), verbose=0)
        oof_cat[seed_idx, val_idx] = cat_model.predict_proba(X[val_idx])[:, 1]
        test_preds_cat[seed_idx] += cat_model.predict_proba(X_test)[:, 1] / N_FOLDS
        
        # MLP
        mlp_params_fold = {**mlp_params, 'random_state': seed}
        mlp_model = MLPClassifier(**mlp_params_fold)
        mlp_model.fit(X_scaled[train_idx], y[train_idx])
        oof_mlp[seed_idx, val_idx] = mlp_model.predict_proba(X_scaled[val_idx])[:, 1]
        test_preds_mlp[seed_idx] += mlp_model.predict_proba(X_test_scaled)[:, 1] / N_FOLDS
        
        print(f"   Fold {fold+1}: XGB={roc_auc_score(y[val_idx], oof_xgb[seed_idx, val_idx]):.6f} | "
              f"LGB={roc_auc_score(y[val_idx], oof_lgb[seed_idx, val_idx]):.6f} | "
              f"CAT={roc_auc_score(y[val_idx], oof_cat[seed_idx, val_idx]):.6f} | "
              f"MLP={roc_auc_score(y[val_idx], oof_mlp[seed_idx, val_idx]):.6f}")
        
        del xgb_model, lgb_model, cat_model, mlp_model
        gc.collect()

# Average predictions across seeds
oof_xgb_avg = oof_xgb.mean(axis=0)
oof_lgb_avg = oof_lgb.mean(axis=0)
oof_cat_avg = oof_cat.mean(axis=0)
oof_mlp_avg = oof_mlp.mean(axis=0)

test_xgb_avg = test_preds_xgb.mean(axis=0)
test_lgb_avg = test_preds_lgb.mean(axis=0)
test_cat_avg = test_preds_cat.mean(axis=0)
test_mlp_avg = test_preds_mlp.mean(axis=0)

print(f"\nOOF Scores (seed-averaged):")
print(f"   XGBoost:  {roc_auc_score(y, oof_xgb_avg):.6f}")
print(f"   LightGBM: {roc_auc_score(y, oof_lgb_avg):.6f}")
print(f"   CatBoost: {roc_auc_score(y, oof_cat_avg):.6f}")
print(f"   MLP:      {roc_auc_score(y, oof_mlp_avg):.6f}")

## SECTION 8: MULTI-LEVEL ENSEMBLE

In [ ]:
# Prepare base predictions
oof_list = [oof_xgb_avg, oof_lgb_avg, oof_cat_avg, oof_mlp_avg]
test_list = [test_xgb_avg, test_lgb_avg, test_cat_avg, test_mlp_avg]
model_names = ['XGBoost', 'LightGBM', 'CatBoost', 'MLP']

print("Base Model OOF Scores:")
for name, oof in zip(model_names, oof_list):
    print(f"   {name}: {roc_auc_score(y, oof):.6f}")

In [ ]:
# Simple average
oof_simple = np.mean(oof_list, axis=0)
test_simple = np.mean(test_list, axis=0)
print(f"Simple Average OOF AUC: {roc_auc_score(y, oof_simple):.6f}")

# Rank average
def rank_average(preds_list):
    ranks = [rankdata(p) for p in preds_list]
    avg_rank = np.mean(ranks, axis=0)
    return avg_rank / len(avg_rank)

oof_rank = rank_average(oof_list)
test_rank = rank_average(test_list)
print(f"Rank Average OOF AUC: {roc_auc_score(y, oof_rank):.6f}")

# Optimized weights
def optimize_weights(oof_list, y_true):
    def objective(weights):
        weights = np.abs(weights) / np.sum(np.abs(weights))
        blend = sum(w * oof for w, oof in zip(weights, oof_list))
        return -roc_auc_score(y_true, blend)
    initial_weights = np.ones(len(oof_list)) / len(oof_list)
    result = minimize(objective, initial_weights, method='Nelder-Mead')
    return np.abs(result.x) / np.sum(np.abs(result.x))

optimal_weights = optimize_weights(oof_list, y)
print(f"Optimal Weights: {dict(zip(model_names, optimal_weights.round(4)))}")

oof_weighted = sum(w * oof for w, oof in zip(optimal_weights, oof_list))
test_weighted = sum(w * pred for w, pred in zip(optimal_weights, test_list))
print(f"Weighted Average OOF AUC: {roc_auc_score(y, oof_weighted):.6f}")

In [ ]:
# Stacking
def stacking_ensemble(oof_list, test_list, y_true, n_splits=5, seed=42):
    oof_stack = np.column_stack(oof_list)
    test_stack = np.column_stack(test_list)
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_meta = np.zeros(len(y_true))
    test_meta = np.zeros(len(test_stack))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(oof_stack, y_true)):
        meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=seed)
        meta_model.fit(oof_stack[train_idx], y_true[train_idx])
        oof_meta[val_idx] = meta_model.predict_proba(oof_stack[val_idx])[:, 1]
        test_meta += meta_model.predict_proba(test_stack)[:, 1] / n_splits
    
    return oof_meta, test_meta

oof_stack, test_stack = stacking_ensemble(oof_list, test_list, y)
print(f"Stacking OOF AUC: {roc_auc_score(y, oof_stack):.6f}")

# Final blend
oof_final = 0.5 * oof_stack + 0.5 * oof_rank
test_final = 0.5 * test_stack + 0.5 * test_rank
print(f"Final Blend OOF AUC: {roc_auc_score(y, oof_final):.6f}")

In [ ]:
# Compare all ensembles
print("\nENSEMBLE COMPARISON")
print("="*60)

ensembles = {
    'Simple Average': (roc_auc_score(y, oof_simple), test_simple),
    'Rank Average': (roc_auc_score(y, oof_rank), test_rank),
    'Weighted Average': (roc_auc_score(y, oof_weighted), test_weighted),
    'Stacking': (roc_auc_score(y, oof_stack), test_stack),
    'Final Blend': (roc_auc_score(y, oof_final), test_final),
}

sorted_ensembles = sorted(ensembles.items(), key=lambda x: -x[1][0])

print(f"\n{'Method':<20} {'OOF AUC':<12}")
print("-"*32)
for name, (score, _) in sorted_ensembles:
    print(f"{name:<20} {score:.6f}")

# Select best
best_ensemble_name = sorted_ensembles[0][0]
best_test_preds = sorted_ensembles[0][1][1]
best_score = sorted_ensembles[0][1][0]

print(f"\nSelected: {best_ensemble_name} (OOF AUC: {best_score:.6f})")

## SECTION 9: VISUALIZATIONS

In [ ]:
# Model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_scores = [
    roc_auc_score(y, oof_xgb_avg),
    roc_auc_score(y, oof_lgb_avg),
    roc_auc_score(y, oof_cat_avg),
    roc_auc_score(y, oof_mlp_avg),
    best_score
]
model_names_plot = ['XGBoost', 'LightGBM', 'CatBoost', 'MLP', 'Final Ensemble']
colors = ['#42a5f5', '#66bb6a', '#ffa726', '#9c27b0', '#ff6b6b']

bars = axes[0].bar(model_names_plot, model_scores, color=colors, edgecolor='black')
axes[0].set_ylabel('ROC-AUC Score')
axes[0].set_title('Model Comparison (OOF)', fontsize=14, fontweight='bold')
axes[0].set_ylim(min(model_scores) - 0.005, max(model_scores) + 0.005)
for bar, score in zip(bars, model_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
               f'{score:.5f}', ha='center', fontweight='bold')

# ROC curves
for name, oof_preds, color in zip(model_names_plot, 
    [oof_xgb_avg, oof_lgb_avg, oof_cat_avg, oof_mlp_avg, oof_final], colors):
    fpr, tpr, _ = roc_curve(y, oof_preds)
    auc = roc_auc_score(y, oof_preds)
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={auc:.5f})', color=color, linewidth=2)

axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/model_comparison_v3.png', dpi=150, bbox_inches='tight')
plt.show()

## SECTION 10: SUBMISSION GENERATION

In [ ]:
# Create submission
submission = sample_sub.copy()
submission['Churn'] = best_test_preds

submission.to_csv(f'{OUTPUT_DIR}/submission_v3.csv', index=False)

print(f"Submission saved to {OUTPUT_DIR}/submission_v3.csv")
print(f"   Shape: {submission.shape}")
print(f"   Selected method: {best_ensemble_name}")
print(f"   OOF AUC: {best_score:.6f}")

# Prediction distribution
fig, ax = plt.subplots(figsize=(8, 4))
submission['Churn'].hist(bins=50, ax=ax, color='coral', edgecolor='black')
ax.set_xlabel('Predicted Churn Probability')
ax.set_ylabel('Count')
ax.set_title('Distribution of Predicted Probabilities (V3)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/prediction_distribution_v3.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPrediction Distribution:")
print(submission['Churn'].describe().round(4))

# Save all predictions
all_preds = pd.DataFrame({
    'id': test_fe['id'],
    'simple_avg': test_simple,
    'rank_avg': test_rank,
    'weighted_avg': test_weighted,
    'stacking': test_stack,
    'final_blend': test_final,
})
all_preds.to_csv(f'{OUTPUT_DIR}/all_predictions_v3.csv', index=False)
print(f"\nAll predictions saved to {OUTPUT_DIR}/all_predictions_v3.csv")

## SECTION 11: SUMMARY

### V3 Improvements Summary

| Component | Improvement |
|-----------|------------|
| **Data** | Original Telco data merged (if available) |
| **Features** | 110+ features |
| **Models** | XGBoost + LightGBM + CatBoost + MLP |
| **Seeds** | 3 seeds per model (12 total models) |
| **Ensemble** | Stacking + Rank Averaging + Blend |
| **Regularization** | Stronger params (fix overfitting) |

### Expected Results

| Metric | Value |
|--------|-------|
| OOF Score | ~0.917 |
| Expected LB | 0.916-0.918 |
| Target | Top 10% |

---

**Competition:** Playground Series S6E3 - Customer Churn Prediction
**Author:** Pratik
**Version:** 3.0
**Date:** 2026-03-26